In [ ]:
import networkx as nx
import numpy as np
import functools

In [ ]:
def threshold_lambda_c(G):
    """
    计算SI模型在网络上的传播阈值 lambda_c
    :param graph: 网络图（networkx.Graph）
    :return: 传播阈值 lambda_c
    """
    # 获取网络的度序列
    degrees = [degree for node, degree in G.degree()]

    # 计算 <k> 和 <k^2>
    k_mean = np.mean(degrees)  # 平均度 <k>
    k2_mean = np.mean([k**2 for k in degrees])  # 度的平方的平均值 <k^2>

    # 根据公式计算传播阈值
    lambda_c = k_mean / k2_mean 
    
    return lambda_c

In [ ]:
@functools.lru_cache(maxsize=None)  # 没有缓存限制
def shortest_path_from_node(graph, source) -> dict['Node' : 'distance']:
    '''
    计算源节点节点source到其他节点的最短路径，包括源节点本身
    返回值为dict[source node to target node, distance]
    '''
    
    return nx.shortest_path_length(graph, source)

In [ ]:
def propagation_probability_model(graph: nx.Graph, i: 'node') -> float:
    # 计算传播阈值
    belta = threshold_lambda_c(graph)
    
    # 获取节点 i 到其他节点的最短路径
    shortest_paths_from_i = shortest_path_from_node(graph, i)

    # 提取不同层级的邻居节点
    layer_1 = set(node for node, distance in shortest_paths_from_i.items() if distance == 1)
    layer_2 = set(node for node, distance in shortest_paths_from_i.items() if distance == 2)
    layer_3 = set(node for node, distance in shortest_paths_from_i.items() if distance == 3)
    
    # 如果没有 1 层邻居，直接返回 0
    if not layer_1:
        return 0.0
    
    # 计算第一层的传播概率
    score_1 = len(layer_1) * belta
    
    # 如果没有 2 层邻居，返回第一层的传播概率
    if not layer_2:
        return score_1
    
    # 计算第二层的传播概率
    propagated_2 = {}  # 存储每个 2 跳邻居节点的传播概率
    for node in layer_2:
        # 计算该节点与 1 跳邻居的交集数量
        number_path2 = len(layer_1 & set(graph.neighbors(node)))
        # 计算未被传播的概率
        unpro_2 = (1 - belta ** 2) ** number_path2
        # 计算被传播的概率
        propagated_2[node] = 1 - unpro_2
    
    score_2 = sum(propagated_2.values())
    
    # 如果没有 3 层邻居，返回第一层和第二层的传播概率之和
    if not layer_3:
        return score_1 + score_2
    
    # 计算第三层的传播概率
    propagated_3 = {}  # 存储每个 3 跳邻居节点的传播概率
    for node in layer_3:
        # 获取与该节点最近的 2 跳邻居节点
        latest_node = layer_2 & set(graph.neighbors(node))
        # 计算未被传播的概率
        unpropagated_3 = 1.0
        for node_2 in latest_node:
            unpropagated_current_path = 1 - propagated_2[node_2] * belta
            unpropagated_3 *= unpropagated_current_path
        # 计算被传播的概率
        propagated_3[node] = 1 - unpropagated_3
    
    score_3 = sum(propagated_3.values())
    
    # 返回所有层级的传播概率之和
    return score_1 + score_2 + score_3

In [ ]:
def propagation_capability(graph:nx.Graph)->dict['node',float]:
    pc = {}
    for node in graph.nodes():
        pc[node] = propagation_probability_model(graph, node)

    return pc